In [1]:
# Import packages

from selenium import webdriver
from datetime import datetime
import time
import pandas as pd
import pickle as pk

# Create an 'instance' of the driver.
# This path should be to wherever you downloaded the driver.
driver = webdriver.Chrome(executable_path="/home/zegveld/Desktop/chromedriver")
# A new Chrome (or other browser) window should open up.

# Enter the main page, all condos in Bangkok are grouped by district.
url ='https://www.halodoc.com/cari-dokter/spesialis/obstetrics-and-gynecologist'
driver.get(url)

In [ ]:
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.common.keys import Keys
# need the below imports to work with Explicit wait
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.by import By

browser = webdriver.Firefox()
browser.get('thewebpage')

search = browser.find_element_by_id('getSearch')
search.click()
search.send_keys('searchitem' + Keys.RETURN)

searchitem = browser.find_elements_by_class_name("name")[0]
searchitem.click()

# Here is the logic that we have to update

# Get number of users rather than the users.
userElems = len(browser.find_elements_by_link_text("#/user/"))

# iterate through each user by using the index
  # if you try to use the find_elements as shown in OP, you will get StaleElement Exception
  # because the user elements references will be refreshed when navigated to next page and
  # load back (so we have to find the elements based on index on the page every time)

for userNum in range(1,userElems):
    # this below explicit wait will make sure the script will wait max 30 sec for the next user to be clicked
    user = WebDriverWait(driver,30).until(EC.presence_of_element_located((By.XPATH,"(#/user/)[" + str(userNum) + "]")))
    # scroll user into view
    user.location_once_scrolled_into_view
    # click on user
    user.click()
    # click on follow link
    follow = WebDriverWait(driver,30).until(EC.presence_of_element_located((By.XPATH,"followAction")))
    follow.click()
    # click on browser back button
    browser.back()

In [2]:
# Write function to scrape all links from the webpage.

def get_all_links(driver):
    links = []
    elements = driver.find_elements_by_class_name('col-12.col-md-8.col-xl-9.hospital-list--result.ng-star-inserted a')
    for elem in elements:
        href = elem.get_attribute("href")
        links.append(href)
    return links

In [3]:
# Run and store the links in 'district_links'

start_time = datetime.now() 

district_links=get_all_links(driver)

time_elapsed = datetime.now() - start_time 
print('Time elapsed (hh:mm:ss.ms) {}'.format(time_elapsed))

Time elapsed (hh:mm:ss.ms) 0:00:02.475399


In [4]:
# Check the length of 'district_links', there are 50 districts in Bangkok. Matched with wiki info.
# https://en.wikipedia.org/wiki/List_of_districts_of_Bangkok

len(district_links)

10

In [5]:
# Re-run the function to retrive all condos in each district.
# Append to 'condo_links'.

start_time = datetime.now()
condo_links=[]

for dist in district_links:
    print(len(condo_links),dist)
    driver = webdriver.Chrome(executable_path="/home/zegveld/Desktop/chromedriver")
    driver.implicitly_wait(10)
    driver.get(dist)
    condo_links.append(get_all_links(driver))
    driver.close()
    time_elapsed = datetime.now() - start_time 
    print('Time elapsed (hh:mm:ss.ms) {}'.format(time_elapsed))
print("completed")

0 None


WebDriverException: Message: chrome not reachable


In [ ]:
print(len(condo_links))
print(condo_links)

In [ ]:
# Turn a (nested) python list into a single list, that contains all the elements of sub lists
# Named as 'condo_links_all'

from itertools import chain
condo_links_all=list(chain.from_iterable(condo_links))

In [ ]:
# Check total links in the 'condo_links_all' list. Total 2540 links.

print("Total condos listed = "+str(len(condo_links_all)))

In [ ]:
condo_links_all[0:10]

In [ ]:
# Dump the retrived links to text file.

with open("condo_links_all.txt", "w") as f:
    for s in condo_links_all:
        f.write(str(s) +"\n")
print("completed")